# OMOP Person Table

Transforms FHIR Patient resources into OMOP CDM `person` table.

## Mapping: FHIR Patient → OMOP Person

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| person_id | Patient.id | Hash to integer |
| gender_concept_id | Patient.gender | male→8507, female→8532, other→0 |
| year_of_birth | Patient.birthDate | Extract year |
| month_of_birth | Patient.birthDate | Extract month |
| day_of_birth | Patient.birthDate | Extract day |
| birth_datetime | Patient.birthDate | Full timestamp |
| race_concept_id | Patient.extension[race] | Map to OMOP race concepts |
| ethnicity_concept_id | Patient.extension[ethnicity] | Map to OMOP ethnicity concepts |
| location_id | Patient.address | Reference to location |
| provider_id | Patient.generalPractitioner | Reference to provider |
| care_site_id | Patient.managingOrganization | Reference to care_site |
| person_source_value | Patient.id | Original FHIR ID |
| gender_source_value | Patient.gender | Original gender code |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

SELECT catalog_use, silver_schema, gold_schema;

In [ ]:
-- Create Gold schema if not exists
DECLARE OR REPLACE VARIABLE create_schema_stmt STRING;
SET VARIABLE create_schema_stmt = 'CREATE SCHEMA IF NOT EXISTS ' || catalog_use || '.' || gold_schema;
EXECUTE IMMEDIATE create_schema_stmt;

USE IDENTIFIER(catalog_use || '.' || gold_schema);

## Create Person Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_person_stmt STRING;

SET VARIABLE create_person_stmt = "
CREATE OR REFRESH STREAMING TABLE person (
  -- Primary key
  person_id BIGINT NOT NULL COMMENT 'Unique person identifier (hashed from FHIR Patient.id)'
  
  -- Demographics
  ,gender_concept_id INT NOT NULL COMMENT 'Gender concept: 8507=Male, 8532=Female, 0=Unknown'
  ,year_of_birth INT NOT NULL COMMENT 'Year of birth'
  ,month_of_birth INT COMMENT 'Month of birth'
  ,day_of_birth INT COMMENT 'Day of birth'
  ,birth_datetime TIMESTAMP COMMENT 'Full birth datetime'
  
  -- Race and Ethnicity (US Core extensions)
  ,race_concept_id INT NOT NULL DEFAULT 0 COMMENT 'Race concept from OMOP vocabulary'
  ,ethnicity_concept_id INT NOT NULL DEFAULT 0 COMMENT 'Ethnicity concept: 38003563=Hispanic, 38003564=Not Hispanic'
  
  -- References
  ,location_id BIGINT COMMENT 'Reference to location table'
  ,provider_id BIGINT COMMENT 'Reference to provider table (general practitioner)'
  ,care_site_id BIGINT COMMENT 'Reference to care_site table (managing organization)'
  
  -- Source values for traceability
  ,person_source_value STRING COMMENT 'Original FHIR Patient ID'
  ,gender_source_value STRING COMMENT 'Original gender value from FHIR'
  ,gender_source_concept_id INT DEFAULT 0 COMMENT 'Source concept for gender'
  ,race_source_value STRING COMMENT 'Original race value from FHIR extension'
  ,race_source_concept_id INT DEFAULT 0 COMMENT 'Source concept for race'
  ,ethnicity_source_value STRING COMMENT 'Original ethnicity value from FHIR extension'
  ,ethnicity_source_concept_id INT DEFAULT 0 COMMENT 'Source concept for ethnicity'
  
  -- Lineage
  ,fhir_patient_uuid STRING COMMENT 'Original FHIR Patient UUID from Silver layer'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Person table - Demographics from FHIR Patient resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer person_id from FHIR Patient.id using hash
  ABS(HASH(COALESCE(id::STRING, patient_uuid))) AS person_id
  
  -- Map gender to OMOP concept_id
  ,CASE LOWER(gender::STRING)
    WHEN 'male' THEN 8507
    WHEN 'female' THEN 8532
    WHEN 'other' THEN 0
    WHEN 'unknown' THEN 0
    ELSE 0
  END AS gender_concept_id
  
  -- Extract birth date components
  ,YEAR(TRY_CAST(birthDate::STRING AS DATE)) AS year_of_birth
  ,MONTH(TRY_CAST(birthDate::STRING AS DATE)) AS month_of_birth
  ,DAY(TRY_CAST(birthDate::STRING AS DATE)) AS day_of_birth
  ,TRY_CAST(birthDate::STRING AS TIMESTAMP) AS birth_datetime
  
  -- Race and ethnicity (from US Core extensions if available)
  -- Default to 0 (Unknown) - extend this logic for US Core race/ethnicity extensions
  ,0 AS race_concept_id
  ,0 AS ethnicity_concept_id
  
  -- References (to be populated when location/provider/care_site tables exist)
  ,NULL AS location_id
  ,NULL AS provider_id
  ,NULL AS care_site_id
  
  -- Source values
  ,id::STRING AS person_source_value
  ,gender::STRING AS gender_source_value
  ,0 AS gender_source_concept_id
  ,NULL AS race_source_value
  ,0 AS race_source_concept_id
  ,NULL AS ethnicity_source_value
  ,0 AS ethnicity_source_concept_id
  
  -- Lineage
  ,patient_uuid AS fhir_patient_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".patient)
WHERE id IS NOT NULL
";

SELECT create_person_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_person_stmt;

In [ ]:
-- Verify person table
SELECT 
  person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  person_source_value,
  gender_source_value
FROM person
LIMIT 10;

In [ ]:
-- Gender distribution
SELECT 
  CASE gender_concept_id
    WHEN 8507 THEN 'Male'
    WHEN 8532 THEN 'Female'
    ELSE 'Unknown/Other'
  END AS gender,
  COUNT(*) AS count
FROM person
GROUP BY gender_concept_id
ORDER BY count DESC;

In [ ]:
-- Age distribution
SELECT 
  FLOOR((YEAR(CURRENT_DATE()) - year_of_birth) / 10) * 10 AS age_decade,
  COUNT(*) AS count
FROM person
WHERE year_of_birth IS NOT NULL
GROUP BY age_decade
ORDER BY age_decade;